# Ispezione dei dati: GOD (.npz, .h5), NSD demo (.pt), NSD grezzi (.mgh)

Notebook per guardare velocemente struttura, forme, tipi e intestazioni dei file usati
nella riproduzione di B2B, e per confrontare i file della demo con quelli ricostruiti.

- Ogni sezione è indipendente: si può eseguire solo quella che serve.
- I file grandi (.h5, .mgh) **non** vengono caricati interi: si leggono intestazioni e piccole porzioni.
- I percorsi sono nella cella *Percorsi*: modificali lì.

**Kernel:** serve un ambiente con `numpy`, `h5py`, `torch`, `nibabel`, `matplotlib`, `ipykernel`.
La cella successiva dice cosa manca.

In [ ]:
import os, glob, importlib
import numpy as np

# Controllo dei pacchetti: le sezioni che ne hanno bisogno si fermano con un messaggio chiaro
PKG = {}
for name in ['h5py', 'torch', 'nibabel', 'matplotlib']:
    try:
        PKG[name] = importlib.import_module(name)
        print('%-11s %s' % (name, getattr(PKG[name], '__version__', 'ok')))
    except ImportError:
        PKG[name] = None
        print('%-11s MANCANTE' % name)

def need(name):
    if PKG.get(name) is None:
        raise ImportError('Questa sezione richiede %s: installalo nel kernel (pip install %s)' % (name, name))
    return PKG[name]

np.set_printoptions(precision=4, suppress=True, linewidth=120)

## Percorsi

In [ ]:
SCRATCH = '/scratch_share/islab/glauco.rinaldi'
REPO    = '/home/g.rinaldi-thesis/projects/Brain-Transformation-with-No-Shared-Data'

DEMO_GOD = SCRATCH + '/demo_data/Processed_data/GOD'   # file della demo (riferimento)
DEMO_NSD = SCRATCH + '/demo_data/Processed_data/NSD'
HO_GOD   = SCRATCH + '/datasets/GOD_Ho2022'             # fMRI GOD scaricata (Ho et al. 2022)
NSD_RAW  = SCRATCH + '/datasets/NSD'                    # beta NSD scaricati (.mgh)

for p in [DEMO_GOD, DEMO_NSD, HO_GOD, NSD_RAW]:
    print('ok ' if os.path.isdir(p) else 'NON TROVATA', p)

## Funzioni di descrizione

`describe(nome, x)` stampa forma, tipo e statistiche di qualunque array (numpy, tensore torch, lista).
Le statistiche si calcolano al massimo su un milione di valori, per non bloccare il kernel.

In [ ]:
def to_numpy(x):
    torch = PKG.get('torch')
    if torch is not None and torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)

def describe(name, x, max_vals=1_000_000):
    a = to_numpy(x)
    line = '%-40s shape=%-22s dtype=%-8s' % (name, str(a.shape), a.dtype)
    if a.size and (np.issubdtype(a.dtype, np.number) or a.dtype == bool):
        flat = a.reshape(-1)
        if flat.size > max_vals:                       # campione uniforme sui file grandi
            flat = flat[:: flat.size // max_vals + 1]
        f = flat.astype(np.float64)
        line += ' min=%.4g max=%.4g mean=%.4g std=%.4g nan=%d' % (
            np.nanmin(f), np.nanmax(f), np.nanmean(f), np.nanstd(f), int(np.isnan(f).sum()))
        if a.dtype == bool or set(np.unique(flat[:10000]).tolist()) <= {0, 1}:
            line += ' | veri=%d' % int(np.count_nonzero(a))
    print(line)
    return a

def compare(name, a, b):
    """Confronto tra due array: identici? differenza massima e media."""
    a, b = to_numpy(a), to_numpy(b)
    if a.shape != b.shape:
        print('%-30s FORME DIVERSE %s vs %s' % (name, a.shape, b.shape)); return
    d = np.abs(a.astype(np.float64) - b.astype(np.float64))
    print('%-30s identici=%s  diff max=%.4g  diff media=%.4g  uguali=%.2f%%'
          % (name, np.array_equal(a, b), np.nanmax(d), np.nanmean(d), 100 * (d == 0).mean()))

## 1. GOD — immagini `.npz`

`images_112.npz` (chiavi `train_images`, `test_images`) e `ext_images_test_112.npz` (chiave `img_112`).
Un `.npz` è un archivio di array numpy: `np.load` lo apre e legge ogni array solo quando lo si chiede.

In [ ]:
def show_npz(path):
    print('==', path)
    with np.load(path) as f:
        for k in f.files:
            describe('  ' + k, f[k])

for name in ['images_112.npz', 'ext_images_test_112.npz']:
    show_npz(os.path.join(DEMO_GOD, name))

In [ ]:
# Qualche immagine affiancata (prima riga: train, seconda: test)
plt = need('matplotlib').pyplot
with np.load(os.path.join(DEMO_GOD, 'images_112.npz')) as f:
    tr, te = f['train_images'], f['test_images']
idx = [0, 1, 2, 3, 4, 5]
fig, ax = plt.subplots(2, len(idx), figsize=(2 * len(idx), 4.4))
for j, i in enumerate(idx):
    ax[0, j].imshow(tr[i]); ax[0, j].set_title('train %d' % i, fontsize=8)
    ax[1, j].imshow(te[i]); ax[1, j].set_title('test %d' % i, fontsize=8)
for a in ax.ravel(): a.axis('off')
plt.tight_layout()

In [ ]:
# Confronto con una mia ricostruzione (es. prodotta da preproc/build_god_images.py)
MIO_NPZ = SCRATCH + '/processed/GOD/images_112.npz'    # <- cambia qui
if os.path.exists(MIO_NPZ):
    with np.load(MIO_NPZ) as m, np.load(os.path.join(DEMO_GOD, 'images_112.npz')) as d:
        for k in ['train_images', 'test_images']:
            compare(k, m[k], d[k])
else:
    print('non trovato:', MIO_NPZ)

## 2. GOD — fMRI `.h5` (formato BData)

Un file BData contiene:
- `dataset`: matrice *trial × colonne*. Le colonne sono i voxel più alcune colonne di metadati (sessione, run, blocco, label…);
- `metadata/key`, `metadata/description`: nome e descrizione di ogni riga di metadati;
- `metadata/value`: matrice *chiavi × colonne*. Per la riga di una chiave, le colonne con valore 1 sono quelle che le appartengono (es. la chiave `VoxelData` marca le colonne dei voxel).

`h5_tree` mostra tutto l'albero del file; `bdata_summary` dice quante colonne appartengono a ogni chiave.
Il codice degli autori usa le righe di metadati 0-4 e 22-28 per posizione: la colonna `riga` serve a controllarle.

In [ ]:
def h5_tree(path):
    h5py = need('h5py')
    print('==', path)
    with h5py.File(path, 'r') as f:
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                print('  %-45s %-18s %s' % (name, str(obj.shape), obj.dtype))
            else:
                print('  %s/' % name)
            for k, v in obj.attrs.items():
                print('      @%s = %r' % (k, v))
        f.visititems(visit)

def _str(x):
    return x.decode() if isinstance(x, bytes) else str(x)

def bdata_summary(path, show_desc=True):
    h5py = need('h5py')
    with h5py.File(path, 'r') as f:
        keys = [_str(k) for k in f['metadata/key'][()]]
        desc = [_str(d) for d in f['metadata/description'][()]]
        val  = f['metadata/value'][()]
        print('dataset', f['dataset'].shape, '| metadata/value', val.shape)
        print('%4s %-28s %8s  %s' % ('riga', 'chiave', 'colonne', 'descrizione' if show_desc else ''))
        for i, (k, d) in enumerate(zip(keys, desc)):
            n = int(np.nansum(val[i] == 1))
            print('%4d %-28s %8d  %s' % (i, k, n, d[:70] if show_desc else ''))
    return keys, val

h5_tree(os.path.join(HO_GOD, 'sub-04_NaturalImageTraining.h5'))

In [ ]:
keys, val = bdata_summary(os.path.join(HO_GOD, 'sub-04_NaturalImageTraining.h5'))

In [ ]:
def bdata_column(path, key):
    """Le colonne di 'dataset' che appartengono alla chiave indicata (es. 'Label', 'Session', 'VoxelData')."""
    h5py = need('h5py')
    with h5py.File(path, 'r') as f:
        keys = [_str(k) for k in f['metadata/key'][()]]
        cols = np.where(f['metadata/value'][keys.index(key)] == 1)[0]
        return f['dataset'][:, cols] if len(cols) < 50 else f['dataset'][:, cols[:50]]   # i voxel: solo i primi 50

p = os.path.join(HO_GOD, 'sub-04_NaturalImageTraining.h5')
for k in ['Session', 'Run', 'Block', 'Label']:
    try:
        a = describe(k, bdata_column(p, k))
        print('   prime righe:', a[:5].ravel())
    except ValueError:
        print(k, ': chiave non presente')
describe('VoxelData (primi 50 voxel)', bdata_column(p, 'VoxelData'))

In [ ]:
# Tutti i soggetti: forma di dataset e numero di voxel (Training e Test)
h5py = need('h5py')
for s in range(1, 6):
    for part in ['Training', 'Test']:
        p = os.path.join(HO_GOD, 'sub-0%d_NaturalImage%s.h5' % (s, part))
        if not os.path.exists(p):
            print('manca', p); continue
        with h5py.File(p, 'r') as f:
            keys = [_str(k) for k in f['metadata/key'][()]]
            nvox = int((f['metadata/value'][keys.index('VoxelData')] == 1).sum())
            print('sub-0%d %-8s dataset=%-15s voxel=%d' % (s, part, f['dataset'].shape, nvox))

## 3. NSD — file `.pt` della demo

Sono oggetti salvati con `torch.save` (tensori, oppure liste o dizionari di tensori).
`torch.load` li carica interi, ma sono piccoli: i dati fMRI già ridotti a 10.000 voxel.
`show_pt` descrive anche i contenitori annidati.

In [ ]:
def show_pt(path, indent='  '):
    torch = need('torch')
    obj = torch.load(path, map_location='cpu')
    print('==', os.path.basename(path), '->', type(obj).__name__)
    def walk(name, o, ind):
        if isinstance(o, dict):
            print(ind + name, 'dict con %d chiavi' % len(o))
            for k, v in list(o.items())[:20]:
                walk(str(k), v, ind + '  ')
        elif isinstance(o, (list, tuple)) and not np.isscalar(o) and len(o) and not np.isscalar(o[0]):
            print(ind + name, '%s di %d elementi' % (type(o).__name__, len(o)))
            for i, v in enumerate(o[:10]):
                walk('[%d]' % i, v, ind + '  ')
        else:
            describe(ind + name, o)
    walk('', obj, indent)
    return obj

pt_files = sorted(glob.glob(os.path.join(DEMO_NSD, '*.pt')))
print(len(pt_files), 'file .pt')
nsd = {os.path.basename(p): show_pt(p) for p in pt_files}

In [ ]:
# Controlli incrociati tra le maschere: quanti trial in train/val/test, con e senza immagini condivise.
# Le maschere sono vettori booleani (o 0/1) lunghi quanto il numero di trial del soggetto.
for name, obj in nsd.items():
    if 'mask' in name:
        a = to_numpy(obj).ravel()
        if a.size < 100_000:
            print('%-70s lung=%6d  veri=%6d' % (name, a.size, int(np.count_nonzero(a))))

## 4. NSD — beta grezzi `.mgh` (FreeSurfer)

I beta fsaverage di NSD sono un file per emisfero (`lh`/`rh`) e per sessione.
Ogni file contiene *vertici × 1 × 1 × trial*: 163.842 vertici dell'emisfero fsaverage e 750 trial per sessione.
`nibabel` legge l'intestazione subito e i dati solo quando li si chiede (`dataobj`):
qui si leggono solo i primi trial, per non caricare ~0,5 GB per file.

In [ ]:
mgh_files = sorted(glob.glob(os.path.join(NSD_RAW, '**', '*.mgh'), recursive=True))
print(len(mgh_files), 'file .mgh trovati')
# Raggruppati per cartella, per vedere come è organizzato il download
from collections import Counter
for d, n in sorted(Counter(os.path.dirname(p) for p in mgh_files).items()):
    print('%4d  %s' % (n, d))

In [ ]:
def show_mgh(path, n_trials=5):
    nib = need('nibabel')
    img = nib.load(path)                 # legge solo l'intestazione
    h = img.header
    print('==', path)
    print('  forma', img.shape, '| tipo su disco', h.get_data_dtype(), '| scala', h.get_slope_inter())
    print('  dimensione voxel/zooms', h.get_zooms())
    print('  affine:\n', img.affine)
    # Parametri MRI salvati in coda al file (TR, flip angle, TE, TI, FoV), se presenti
    for k in ['tr', 'flip_angle', 'te', 'ti', 'fov']:
        try:
            print('  %-10s %s' % (k, h[k]))
        except Exception:
            pass
    part = np.asanyarray(img.dataobj[..., :n_trials])   # solo i primi trial
    describe('  primi %d trial' % n_trials, part)
    return img

if mgh_files:
    img = show_mgh(mgh_files[0])

In [ ]:
# Panoramica di tutte le sessioni di un soggetto: forma e tipo, senza leggere i dati
nib = need('nibabel')
SUBJ = 'subj01'
for p in [p for p in mgh_files if SUBJ in p][:80]:
    im = nib.load(p)
    print('%-45s %-22s %s' % (os.path.basename(p), im.shape, im.header.get_data_dtype()))

In [ ]:
# Distribuzione dei valori di un trial su tutti i vertici: utile per capire la scala dei beta
# (i beta NSD sono spesso salvati come interi da dividere per 300 per avere la % di segnale)
plt = need('matplotlib').pyplot
if mgh_files:
    v = np.asanyarray(nib.load(mgh_files[0]).dataobj[..., 0]).ravel().astype(np.float64)
    describe('trial 0', v)
    plt.figure(figsize=(6, 3)); plt.hist(v[np.isfinite(v)], bins=200); plt.yscale('log')
    plt.title(os.path.basename(mgh_files[0]) + ' — trial 0'); plt.tight_layout()